# Eff-3DPSeg (spconv) on Google Colab

Before running: **Runtime -> Change runtime type -> GPU (T4)**. spconv needs CUDA.

Pheno4D comes from your Kaggle dataset. Add your Kaggle API key as Colab *Secrets* (key icon in the left bar) named `KAGGLE_USERNAME` and `KAGGLE_KEY`, with notebook access on. Never paste the key into a cell that you save or share.

In [ ]:
# ---- edit these ----
REPO_URL = 'https://github.com/gitmate21/3d-phenomics.git'
KAGGLE_DATASET = 'hsenagb/pheno4d-clone'      # owner/name of your Kaggle dataset
# --------------------
import os, torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())
if not torch.cuda.is_available() or torch.version.cuda is None:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> GPU, then Runtime -> Restart session.')

# Google Drive keeps checkpoints and logs if the session is cut off.
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/eff3dpseg/checkpoints'
LOG_DIR  = '/content/drive/MyDrive/eff3dpseg/logs'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
# Clone, or update an existing clone, then enter the project folder.
REPO = '/content/3d-phenomics'
if os.path.isdir(REPO):
    !git -C {REPO} pull origin main
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}/eff3dpseg

In [ ]:
# spconv wheels are per CUDA version: pick the one matching torch.
import subprocess, sys
try:
    import spconv.pytorch  # noqa
    print('spconv already installed')
except ImportError:
    pkg = 'spconv-cu' + torch.version.cuda.replace('.', '')
    print('installing', pkg)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    if r.returncode != 0:
        raise SystemExit(f'No spconv wheel for CUDA {torch.version.cuda} / Python {sys.version_info[:2]}. '
                         'Try the nearest spconv-cuXXX listed at https://github.com/traveller59/spconv#install')
import spconv
print('spconv', spconv.__version__)

In [ ]:
# Download Pheno4D from Kaggle (needs the Secrets described above).
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
!pip install -q kaggle
RAW_ROOT = '/content/pheno4d_raw'
if not os.path.isdir(RAW_ROOT):
    !kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_ROOT} --unzip
!ls {RAW_ROOT}

In [ ]:
# Point RAW_DIR at the folder that contains one sub-folder per plant (look at the listing above).
RAW_DIR = RAW_ROOT + '/Pheno4D'
!ls {RAW_DIR}
!ls {RAW_DIR}/$(ls {RAW_DIR} | head -1) | head

In [ ]:
!python scripts/smoke_test.py

In [ ]:
# Convert raw txt -> npz + plant-level splits. Hold out whole plants for val/test:
# edit the plant names to ones that exist in the listing above.
DATA = '/content/pheno4d_npz'
!python scripts/prepare_pheno4d.py --raw {RAW_DIR} --out {DATA} --val-plants Tomato06 --test-plants Tomato07

In [ ]:
# --voxel-size is in the raw point-cloud units; the original used 0.2. Check your units!
# Re-run this same cell after a disconnect: --resume auto continues from the Drive checkpoint.
!python scripts/train.py --data {DATA} --run-name demo --max-iter 2000 --validate-step 50 \
    --ckpt-dir {CKPT_DIR} --log-dir {LOG_DIR} --resume auto --time-limit-hours 10

In [ ]:
!python scripts/infer.py --data {DATA} --ckpt {CKPT_DIR}/demo_best.pth --phase test \
    --out-dir /content/drive/MyDrive/eff3dpseg/preds --save-ply